# Global MultiModalSoilNet — Dataset Compilation and Alignment

**Purpose:** compile the global tabular features, target observations, static attributes, and verified annual raster row mappings into a stable format for a separate model-training/evaluation notebook.

## Design decisions

- One `sample_index.parquet` row per canonical `sample_id`.
- All in-situ target observations are preserved in `target_observations.parquet`. This notebook also creates `daily_targets.parquet` using Option A: the mean of valid linked observations per `sample_id`, including across sensors/depths. The aggregation is explicit and its depth-mixing metadata is retained.
- Raster patches stay in the original yearly Zarr stores. The index contains row pointers into those stores, so the notebook never loads all patches into memory.
- No feature imputation or normalization, train/test split, residual trimming, or calibration is applied here. Target aggregation is separately defined, auditable, and does not overwrite the source observations. Those choices must be made after defining the target and split, using training-only statistics.
- `SMAP` and its conflict/observation metadata are preserved. A conflict flag is not used to drop a row.
- EVI, LST and Sentinel-1 are treated as separate spatial modalities because EVI is 27×27 while LST and Sentinel-1 are 13×13.

The notebook is intended to run from your project venv on the machine holding `~/Code/UG_SM/Data/Global`.

## Output contract

The output folder is `Data/Global/model_ready_global_v1/`:

| File | Contents |
|---|---|
| `sample_index.parquet` | One row per candidate `sample_id`, tabular features, quality flags, and MODIS/Sentinel-1 row indices |
| `target_observations.parquet` | All linked in-situ target rows with sensor/depth/source metadata preserved (unchanged by aggregation) |
| `daily_targets.parquet` | One row per `sample_id`; mean of valid linked observations plus counts, range, sensor and depth QC metadata |
| `target_sample_summary.parquet` | QC-only summary of label counts/ranges per `sample_id` (retained for diagnostics) |
| `raster_alignment_audit.parquet` | One QA record per year and raster family |
| `feature_schema.json` | Explicit spatial, temporal, static, anchor, metadata and target contracts |
| `prep_qc.json` | Counts, join coverage, missingness and alignment summary |
| `README.md` | How the training notebook should consume the outputs |

The original Zarr arrays are referenced by a relative path template and row index; they are not copied. Keep the original `Data/Global` directory intact.

In [1]:
from __future__ import annotations

import json
import math
import shutil
import warnings
from pathlib import Path
from typing import Any

import duckdb
import numpy as np
import pandas as pd
import pyarrow  # noqa: F401 - confirms parquet engine is available
import zarr
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("duckdb:", duckdb.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("zarr:", zarr.__version__)

duckdb: 1.5.6
pandas: 2.3.3
numpy: 2.4.4
zarr: 3.4.0


In [2]:
# ========================= CONFIG =========================
# This resolves to /home/<user>/Code/UG_SM/Data/Global on your Linux setup.
ROOT = Path.home() / "Code" / "UG_SM" / "Data" / "Global"
OUTPUT_DIR = ROOT / "model_ready_global_v1"
STAGING_DIR = OUTPUT_DIR / "_build_tmp"
YEARS = list(range(2015, 2027))

# Set True only when you intentionally want to replace the existing compiled outputs.
REBUILD_OUTPUT = True

CANDIDATES_PATH = ROOT / "station_date_candidates.parquet"
LABELS_PATH = ROOT / "global_ismn_daily.parquet"
STATIONS_PATH = ROOT / "stations.parquet"
ERA5_PATH = ROOT / "era5_backend" / "era5_sample_features.parquet"
SMAP_PATH = ROOT / "smap" / "smap_by_sample.parquet"
STATIC_SOIL_PATH = ROOT / "Statics" / "global_static_soil.csv"
STATIC_TERRAIN_PATH = ROOT / "Statics" / "global_static_terrain.csv"
S1_ROOT = ROOT / "sentinel1" / "nearest_valid_overview_v1"
MODIS_ROOT = ROOT / "modis" / "evi_lst_patch_dataset"

# Expected producer contracts confirmed in the prior raster row-order audit.
MODIS_ARRAY_NAMES = {"evi": "evi", "lst": "lst", "done": "done"}
S1_ARRAY_NAMES = {
    "vv": "vv", "vh": "vh", "hh": "hh", "hv": "hv",
    "vv_vh_ratio": "vv_vh_ratio", "source_day_offset": "source_day_offset",
    "obs_count": "obs_count", "done": "sample_done",
}

# Stop instead of producing a silently misaligned index.
FAIL_ON_INCOMPLETE_RASTERS = True

print("ROOT:", ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

ROOT: /home/mayur/Code/UG_SM/Data/Global
OUTPUT_DIR: /home/mayur/Code/UG_SM/Data/Global/model_ready_global_v1


In [3]:
# ========================= INPUT FILE CHECKS =========================
required_paths = [
    CANDIDATES_PATH, LABELS_PATH, STATIONS_PATH, ERA5_PATH, SMAP_PATH,
    STATIC_SOIL_PATH, STATIC_TERRAIN_PATH,
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required input files are missing. Check ROOT/config above:\n" + "\n".join(missing_paths)
    )

if OUTPUT_DIR.exists() and not REBUILD_OUTPUT:
    existing_outputs = [p.name for p in OUTPUT_DIR.iterdir() if p.is_file() and p.suffix in {".parquet", ".json"}]
    if existing_outputs:
        raise FileExistsError(
            f"Compiled outputs already exist under {OUTPUT_DIR}. Set REBUILD_OUTPUT=True to replace them intentionally."
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if STAGING_DIR.exists():
    shutil.rmtree(STAGING_DIR)
STAGING_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute("SET preserve_insertion_order = true")

def sql_path(path: Path) -> str:
    """Single-quote-safe absolute path for DuckDB SQL literals."""
    return str(path.resolve()).replace("'", "''")

def parquet_scan(path: Path) -> str:
    return f"read_parquet('{sql_path(path)}')"

def scalar(sql: str) -> Any:
    return con.execute(sql).fetchone()[0]

def describe_parquet(path: Path) -> pd.DataFrame:
    return con.execute(f"DESCRIBE SELECT * FROM {parquet_scan(path)}").df()

for label, path in [
    ("candidates", CANDIDATES_PATH), ("labels", LABELS_PATH), ("stations", STATIONS_PATH),
    ("ERA5", ERA5_PATH), ("SMAP", SMAP_PATH),
]:
    print(f"\n--- {label}: {path.relative_to(ROOT)} ---")
    print(describe_parquet(path).to_string(index=False))


--- candidates: station_date_candidates.parquet ---
     column_name column_type null  key default extra
       sample_id      BIGINT  YES None    None  None
      station_id     VARCHAR  YES None    None  None
     sample_date        DATE  YES None    None  None
         network     VARCHAR  YES None    None  None
    network_name     VARCHAR  YES None    None  None
         station     VARCHAR  YES None    None  None
        latitude      DOUBLE  YES None    None  None
       longitude      DOUBLE  YES None    None  None
     elevation_m      DOUBLE  YES None    None  None
    climate_zone     VARCHAR  YES None    None  None
source_row_count      BIGINT  YES None    None  None
min_depth_from_m      DOUBLE  YES None    None  None
  max_depth_to_m      DOUBLE  YES None    None  None
         sensors     VARCHAR  YES None    None  None
   depth_classes     VARCHAR  YES None    None  None

--- labels: global_ismn_daily.parquet ---
        column_name column_type null  key default extra


In [4]:
# ========================= KEY / SCHEMA VALIDATION =========================
# Required identity columns from the agreed global schema.
candidate_cols = set(describe_parquet(CANDIDATES_PATH)["column_name"])
label_cols = set(describe_parquet(LABELS_PATH)["column_name"])
era5_cols = set(describe_parquet(ERA5_PATH)["column_name"])
smap_cols = set(describe_parquet(SMAP_PATH)["column_name"])

required_candidate_cols = {"sample_id", "station_id", "network", "sample_date", "latitude", "longitude"}
required_label_cols = {"station_id", "network", "sample_date", "soil_moisture_m3_m3"}
required_era5_cols = {"sample_id", "station_id", "sample_date", "t2m_K", "d2m_K", "precip_mm", "VPD_kPa", "API_mm"}
required_smap_cols = {"sample_id", "SMAP", "SMAP_n_obs", "SMAP_cell_conflict"}

for label, have, required in [
    ("candidate", candidate_cols, required_candidate_cols),
    ("label", label_cols, required_label_cols),
    ("ERA5", era5_cols, required_era5_cols),
    ("SMAP", smap_cols, required_smap_cols),
]:
    missing = sorted(required - have)
    if missing:
        raise ValueError(f"{label} source lacks required columns: {missing}")

candidate_count = int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(CANDIDATES_PATH)}"))
candidate_distinct = int(scalar(f"SELECT COUNT(DISTINCT sample_id) FROM {parquet_scan(CANDIDATES_PATH)}"))
if candidate_count != candidate_distinct:
    raise ValueError(f"Canonical sample_id is not unique: {candidate_count:,} rows vs {candidate_distinct:,} distinct IDs")

for label, path in [("ERA5", ERA5_PATH), ("SMAP", SMAP_PATH)]:
    rows = int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(path)}"))
    distinct = int(scalar(f"SELECT COUNT(DISTINCT sample_id) FROM {parquet_scan(path)}"))
    if rows != distinct:
        raise ValueError(f"{label} table is not one row per sample_id: {rows:,} rows, {distinct:,} distinct IDs")
    print(f"{label}: {rows:,} rows / {distinct:,} distinct sample IDs")

print({"canonical_rows": candidate_count, "distinct_sample_ids": candidate_distinct})

ERA5: 3,514,643 rows / 3,514,643 distinct sample IDs
SMAP: 3,514,643 rows / 3,514,643 distinct sample IDs
{'canonical_rows': 3514643, 'distinct_sample_ids': 3514643}


## Raster alignment checks

**MODIS row mapping:** the MODIS producer builds each year's `year_samples` by sorting on `(sample_date, sample_id)`, resets the index, and writes patches at those dataframe index positions. However, it writes `samples.parquet` only if that sidecar does not already exist. Therefore the sidecar can have the correct sample keys but stale/different row order. This notebook treats the sidecar order comparison as a diagnostic, verifies sidecar identities by `sample_id`, and constructs `modis_row_index` from the producer's documented `(sample_date, sample_id)` write order. The Zarr arrays do not embed sample IDs, so the physical-order assumption is only valid if the existing arrays were written/resumed using that producer ordering; if the writer's ordering changed between runs, rebuild the MODIS Zarr stores and sidecars together before training.

For Sentinel-1, the sidecar's explicit `row_index` is used, and its key order, uniqueness, bounds and array dimensions are checked.

The checks do not read all raster pixels into memory: only completion flags and a few representative rows are inspected.


In [5]:
# ========================= RASTER INDEX + AUDIT =========================
def find_s1_sidecar(year: int) -> Path:
    options = [
        S1_ROOT / f"samples_nearest_valid_{year}.parquet",
        S1_ROOT / f"samples_{year}.parquet",
        ROOT / "sentinel1" / f"samples_nearest_valid_{year}.parquet",
        ROOT / "sentinel1" / f"samples_{year}.parquet",
    ]
    for p in options:
        if p.exists():
            return p
    raise FileNotFoundError(f"No Sentinel-1 sample sidecar found for {year}; checked: {[str(p) for p in options]}")


def normalize_day(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def open_group(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing Zarr store: {path}")
    # Some Windows-originated files may cause a harmless Zone.Identifier warning.
    return zarr.open_group(str(path), mode="r")


def read_done_all(arr) -> bool:
    values = np.asarray(arr[:])
    return bool(values.size > 0 and np.all(values.astype(bool)))

raster_index_files = []
audit_rows = []

for year in YEARS:
    print(f"\n===== {year} =====")
    expected = con.execute(f"""
        SELECT sample_id, CAST(station_id AS VARCHAR) AS station_id,
               CAST(sample_date AS DATE) AS sample_date
        FROM {parquet_scan(CANDIDATES_PATH)}
        WHERE YEAR(CAST(sample_date AS DATE)) = {int(year)}
        ORDER BY CAST(sample_date AS DATE), sample_id
    """).df()
    expected_ids = expected["sample_id"].to_numpy(dtype=np.int64)
    expected_dates = normalize_day(expected["sample_date"])
    expected_stations = expected["station_id"].astype(str).to_numpy()

    # ---------- MODIS ----------
    modis_sidecar_path = MODIS_ROOT / str(year) / "samples.parquet"
    modis_store_path = MODIS_ROOT / str(year) / "patches.zarr"
    if not modis_sidecar_path.exists():
        raise FileNotFoundError(modis_sidecar_path)
    modis_sidecar = pd.read_parquet(modis_sidecar_path)
    for col in ["sample_id", "station_id", "sample_date"]:
        if col not in modis_sidecar.columns:
            raise ValueError(f"MODIS {year} sidecar missing {col!r}")

    # Validate the sidecar by identity, not by physical order. A previous run can
    # leave samples.parquet in an older order because the producer only writes it
    # when it does not already exist.
    if modis_sidecar["sample_id"].isna().any():
        raise ValueError(f"MODIS {year} sidecar has null sample_id values")
    modis_sidecar["sample_id"] = pd.to_numeric(modis_sidecar["sample_id"], errors="raise").astype(np.int64)
    if modis_sidecar["sample_id"].duplicated().any():
        raise ValueError(f"MODIS {year} sidecar has duplicate sample_id values")

    modis_sidecar_order_match = (
        len(modis_sidecar) == len(expected)
        and np.array_equal(modis_sidecar["sample_id"].to_numpy(dtype=np.int64), expected_ids)
        and np.array_equal(modis_sidecar["station_id"].astype(str).str.strip().to_numpy(), expected_stations.astype(str))
        and np.array_equal(normalize_day(modis_sidecar["sample_date"]).astype("int64").to_numpy(), expected_dates.astype("int64").to_numpy())
    )

    modis_meta = modis_sidecar[["sample_id", "station_id", "sample_date"]].copy()
    modis_meta["station_id"] = modis_meta["station_id"].astype(str).str.strip()
    modis_meta["sample_date"] = normalize_day(modis_meta["sample_date"])
    expected_meta = pd.DataFrame({
        "sample_id": expected_ids,
        "station_id": expected_stations.astype(str),
        "sample_date": expected_dates,
    })
    modis_key_check = modis_meta.merge(
        expected_meta, on="sample_id", how="outer",
        suffixes=("_sidecar", "_canonical"), indicator=True,
        validate="one_to_one",
    )
    modis_sidecar_keys_match = (
        len(modis_sidecar) == len(expected)
        and (modis_key_check["_merge"] == "both").all()
        and (modis_key_check["station_id_sidecar"] == modis_key_check["station_id_canonical"]).all()
        and (modis_key_check["sample_date_sidecar"] == modis_key_check["sample_date_canonical"]).all()
    )
    if not modis_sidecar_keys_match:
        bad = modis_key_check.loc[
            (modis_key_check["_merge"] != "both")
            | (modis_key_check["station_id_sidecar"] != modis_key_check["station_id_canonical"])
            | (modis_key_check["sample_date_sidecar"] != modis_key_check["sample_date_canonical"])
        ].head(5)
        raise ValueError(
            f"MODIS {year} sidecar sample keys do not match canonical sample IDs/metadata. "
            f"Example mismatches: {bad.to_dict(orient='records')}"
        )

    if not modis_sidecar_order_match:
        print(
            f"WARNING MODIS {year}: sidecar row order differs from canonical order, "
            "but sample IDs/station/date match. Mapping rows from the producer's "
            "documented sorted write order, not the sidecar's physical order."
        )

    modis_group = open_group(modis_store_path)
    missing_arrays = [name for name in MODIS_ARRAY_NAMES.values() if name not in modis_group]
    if missing_arrays:
        raise KeyError(f"MODIS {year} store lacks expected arrays: {missing_arrays}; has {list(modis_group.array_keys())}")
    evi = modis_group[MODIS_ARRAY_NAMES["evi"]]
    lst = modis_group[MODIS_ARRAY_NAMES["lst"]]
    modis_done = modis_group[MODIS_ARRAY_NAMES["done"]]
    modis_shapes_match = (
        tuple(evi.shape) == (len(expected), 27, 27)
        and tuple(lst.shape) == (len(expected), 13, 13)
        and tuple(modis_done.shape) == (len(expected),)
    )
    modis_all_done = read_done_all(modis_done)
    modis_reads_ok = True
    try:
        for idx in sorted(set([0, len(expected)//2, len(expected)-1])):
            if idx >= 0:
                _ = np.asarray(evi[idx])
                _ = np.asarray(lst[idx])
    except Exception as exc:
        modis_reads_ok = False
        raise RuntimeError(f"MODIS representative read failed for {year}: {exc}") from exc

    # ---------- Sentinel-1 ----------
    s1_sidecar_path = find_s1_sidecar(year)
    s1_store_path = S1_ROOT / f"s1_nearest_valid_{year}.zarr"
    s1_sidecar = pd.read_parquet(s1_sidecar_path)
    required_s1_cols = {"sample_id", "station_id", "sample_date", "row_index"}
    if not required_s1_cols.issubset(s1_sidecar.columns):
        raise ValueError(f"Sentinel-1 {year} sidecar missing columns: {sorted(required_s1_cols - set(s1_sidecar.columns))}")
    s1_sidecar["row_index"] = pd.to_numeric(s1_sidecar["row_index"], errors="raise").astype(np.int64)
    if s1_sidecar["sample_id"].duplicated().any():
        raise ValueError(f"Sentinel-1 {year} sidecar has duplicate sample_id values")
    if s1_sidecar["row_index"].duplicated().any():
        raise ValueError(f"Sentinel-1 {year} sidecar has duplicate row_index values")
    s1_by_row = s1_sidecar.sort_values("row_index", kind="mergesort").reset_index(drop=True)
    s1_row_index_match = np.array_equal(s1_by_row["row_index"].to_numpy(), np.arange(len(s1_by_row), dtype=np.int64))
    s1_order_contract = [c for c in ["cluster_id", "station_id", "sample_date"] if c in s1_sidecar.columns]
    if len(s1_order_contract) == 3:
        s1_expected_write_order = s1_sidecar.sort_values(s1_order_contract, kind="mergesort").reset_index(drop=True)
        s1_order_match = np.array_equal(
            s1_expected_write_order["sample_id"].to_numpy(dtype=np.int64),
            s1_by_row["sample_id"].to_numpy(dtype=np.int64),
        )
    else:
        # Explicit row_index + key verification still lets us map each sample safely;
        # mark producer sort order as not independently checked in this case.
        s1_order_match = None

    s1_meta = s1_by_row[["sample_id", "station_id", "sample_date"]].copy()
    s1_meta["sample_id"] = s1_meta["sample_id"].astype(np.int64)
    s1_meta["station_id"] = s1_meta["station_id"].astype(str)
    s1_meta["sample_date"] = normalize_day(s1_meta["sample_date"])
    expected_meta = pd.DataFrame({"sample_id": expected_ids, "station_id": expected_stations, "sample_date": expected_dates})
    merged_meta = s1_meta.merge(expected_meta, on="sample_id", how="outer", suffixes=("_s1", "_canonical"), indicator=True, validate="one_to_one")
    s1_key_match = (
        len(s1_sidecar) == len(expected)
        and (merged_meta["_merge"] == "both").all()
        and (merged_meta["station_id_s1"].astype(str) == merged_meta["station_id_canonical"].astype(str)).all()
        and (merged_meta["sample_date_s1"] == merged_meta["sample_date_canonical"]).all()
    )
    if not s1_key_match:
        raise ValueError(f"Sentinel-1 sidecar keys do not exactly match canonical samples for {year}.")

    s1_group = open_group(s1_store_path)
    missing_arrays = [name for name in S1_ARRAY_NAMES.values() if name not in s1_group]
    if missing_arrays:
        raise KeyError(f"Sentinel-1 {year} store lacks expected arrays: {missing_arrays}; has {list(s1_group.array_keys())}")
    s1_shapes_match = all(tuple(s1_group[name].shape)[0] == len(s1_sidecar) for name in S1_ARRAY_NAMES.values())
    s1_done = s1_group[S1_ARRAY_NAMES["done"]]
    s1_all_done = read_done_all(s1_done)
    complete_attr = bool(s1_group.attrs.get("complete", False))
    s1_reads_ok = True
    try:
        for idx in sorted(set([0, len(s1_by_row)//2, len(s1_by_row)-1])):
            if idx >= 0:
                for key in ["vv", "vh", "vv_vh_ratio", "source_day_offset", "obs_count"]:
                    _ = np.asarray(s1_group[S1_ARRAY_NAMES[key]][idx])
    except Exception as exc:
        s1_reads_ok = False
        raise RuntimeError(f"Sentinel-1 representative read failed for {year}: {exc}") from exc

    passed = all([
        modis_sidecar_keys_match, modis_shapes_match, modis_all_done, modis_reads_ok,
        s1_row_index_match, s1_key_match, s1_shapes_match, s1_all_done,
        s1_reads_ok, complete_attr,
    ]) and (s1_order_match is not False)
    if FAIL_ON_INCOMPLETE_RASTERS and not passed:
        raise RuntimeError(f"Raster QA did not pass for {year}; check this year's audit record.")

    # MODIS producer's physical Zarr index is the position in `year_samples`,
    # which is sorted by sample_date then sample_id before patch writes.
    mapping = pd.DataFrame({
        "sample_id": expected_ids,
        "year": np.full(len(expected_ids), year, dtype=np.int16),
        "modis_row_index": np.arange(len(expected_ids), dtype=np.int64),
    })
    s1_mapping = s1_by_row[["sample_id", "row_index"]].rename(columns={"row_index": "s1_row_index"}).copy()
    s1_mapping["sample_id"] = s1_mapping["sample_id"].astype(np.int64)
    mapping = mapping.merge(s1_mapping, on="sample_id", how="inner", validate="one_to_one")
    if len(mapping) != len(expected):
        raise ValueError(f"Raster index has {len(mapping):,} rows but expected {len(expected):,} for {year}")
    mapping_path = STAGING_DIR / f"raster_index_{year}.parquet"
    mapping.to_parquet(mapping_path, index=False, compression="zstd")
    raster_index_files.append(mapping_path)

    audit_rows.append({
        "year": year, "expected_rows": len(expected),
        "modis_rows": len(modis_sidecar),
        "modis_sidecar_order_match": bool(modis_sidecar_order_match),
        "modis_sidecar_keys_match": bool(modis_sidecar_keys_match),
        "modis_mapping_basis": "producer_sorted_sample_date_sample_id_order",
        "modis_shapes_match": bool(modis_shapes_match), "modis_all_done": bool(modis_all_done),
        "modis_reads_ok": bool(modis_reads_ok),
        "s1_rows": len(s1_sidecar), "s1_order_match": s1_order_match,
        "s1_row_index_match": bool(s1_row_index_match), "s1_key_match": bool(s1_key_match),
        "s1_shapes_match": bool(s1_shapes_match), "s1_all_done": bool(s1_all_done),
        "s1_reads_ok": bool(s1_reads_ok), "s1_complete_attr": bool(complete_attr),
        "passed": bool(passed),
    })
    print({"year": year, "rows": len(expected),
           "MODIS_sidecar_order_matches": modis_sidecar_order_match,
           "MODIS_keys_match": modis_sidecar_keys_match,
           "MODIS_shapes_match": modis_shapes_match,
           "MODIS_all_done": modis_all_done,
           "Sentinel1": s1_key_match and s1_row_index_match and s1_shapes_match and s1_all_done and complete_attr,
           "passed": passed})

raster_audit = pd.DataFrame(audit_rows)
display(raster_audit)
if len(raster_audit) != len(YEARS) or not raster_audit["passed"].all():
    raise RuntimeError("Expected every configured year to pass the raster audit.")


===== 2015 =====
WARNING MODIS 2015: sidecar row order differs from canonical order, but sample IDs/station/date match. Mapping rows from the producer's documented sorted write order, not the sidecar's physical order.
{'year': 2015, 'rows': 329903, 'MODIS_sidecar_order_matches': False, 'MODIS_keys_match': np.True_, 'MODIS_shapes_match': True, 'MODIS_all_done': True, 'Sentinel1': True, 'passed': True}

===== 2016 =====
WARNING MODIS 2016: sidecar row order differs from canonical order, but sample IDs/station/date match. Mapping rows from the producer's documented sorted write order, not the sidecar's physical order.
{'year': 2016, 'rows': 394790, 'MODIS_sidecar_order_matches': False, 'MODIS_keys_match': np.True_, 'MODIS_shapes_match': True, 'MODIS_all_done': True, 'Sentinel1': True, 'passed': True}

===== 2017 =====
WARNING MODIS 2017: sidecar row order differs from canonical order, but sample IDs/station/date match. Mapping rows from the producer's documented sorted write order, not t

,year,expected_rows,modis_rows,modis_sidecar_order_match,modis_sidecar_keys_match,modis_mapping_basis,modis_shapes_match,modis_all_done,modis_reads_ok,s1_rows,s1_order_match,s1_row_index_match,s1_key_match,s1_shapes_match,s1_all_done,s1_reads_ok,s1_complete_attr,passed
0,2015,329903,329903,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,329903,True,True,True,True,True,True,True,True
1,2016,394790,394790,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,394790,True,True,True,True,True,True,True,True
2,2017,369953,369953,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,369953,True,True,True,True,True,True,True,True
3,2018,344806,344806,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,344806,True,True,True,True,True,True,True,True
4,2019,308679,308679,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,308679,True,True,True,True,True,True,True,True
5,2020,269853,269853,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,269853,True,True,True,True,True,True,True,True
6,2021,259625,259625,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,259625,True,True,True,True,True,True,True,True
7,2022,260490,260490,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,260490,True,True,True,True,True,True,True,True
8,2023,233307,233307,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,233307,True,True,True,True,True,True,True,True
9,2024,264946,264946,False,True,producer_sorted_sample_date_sample_id_order,True,True,True,264946,True,True,True,True,True,True,True,True


## Static feature joins and sample-index construction

Static tables are joined on the composite station identity `(network, station_id)`. This is safer than joining on `station_id` alone across a global multi-network dataset. The code checks uniqueness of each static table before joining to prevent accidental row multiplication.

Temperature fields are converted from Kelvin to Celsius for readability. Soil fractions remain in their stored percentage units. **No normalization or imputation occurs here.**

In [6]:
# ========================= STATIC TABLE VALIDATION =========================
soil_df = pd.read_csv(STATIC_SOIL_PATH, dtype={"station_key": "string", "station_id": "string", "network": "string"})
terrain_df = pd.read_csv(STATIC_TERRAIN_PATH, dtype={"station_key": "string", "station_id": "string", "network": "string"})

for name, frame, required in [
    ("static soil", soil_df, {"station_id", "network", "clay_pct", "sand_pct", "silt_pct"}),
    ("static terrain", terrain_df, {"station_id", "network", "elevation_m", "slope_deg", "aspect_deg"}),
]:
    miss = sorted(required - set(frame.columns))
    if miss:
        raise ValueError(f"{name} table missing expected columns: {miss}")
    frame["station_id"] = frame["station_id"].astype("string").str.strip()
    frame["network"] = frame["network"].astype("string").str.strip()
    dup = frame.duplicated(["network", "station_id"], keep=False)
    if dup.any():
        examples = frame.loc[dup, ["network", "station_id"]].head(10).to_dict("records")
        raise ValueError(f"{name} table has duplicate (network, station_id) keys. Examples: {examples}")

soil_stage = STAGING_DIR / "static_soil.parquet"
terrain_stage = STAGING_DIR / "static_terrain.parquet"
soil_df[["network", "station_id", "clay_pct", "sand_pct", "silt_pct"]].to_parquet(soil_stage, index=False)
terrain_df[["network", "station_id", "elevation_m", "slope_deg", "aspect_deg"]].to_parquet(terrain_stage, index=False)

print("Static soil rows:", f"{len(soil_df):,}")
print("Static terrain rows:", f"{len(terrain_df):,}")
print("Static soil example:")
display(soil_df.head())
print("Static terrain example:")
display(terrain_df.head())

Static soil rows: 2,138
Static terrain rows: 2,210
Static soil example:


,station_key,station_id,network,clay_pct,sand_pct,silt_pct
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,6.9,83.0,10.1
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,13.7,68.5,17.7
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,14.7,64.7,20.6
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,15.6,69.0,15.4
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,15.1,69.5,15.4


Static terrain example:


,station_key,station_id,network,elevation_m,slope_deg,aspect_deg
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,208.64137,1.303040,351.192230
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,417.68732,3.574303,88.476840
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,422.68190,0.984109,286.395700
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,441.50824,2.156732,353.058700
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,446.75040,2.320163,18.113632


In [7]:
# ========================= JOIN-CARDINALITY CHECKS =========================
# Feature sources must be at most one row per sample_id before joining.
for label, path in [("ERA5", ERA5_PATH), ("SMAP", SMAP_PATH)]:
    dup_n = int(scalar(f"""
        SELECT COUNT(*) FROM (
            SELECT sample_id FROM {parquet_scan(path)}
            GROUP BY sample_id HAVING COUNT(*) > 1
        )
    """))
    if dup_n:
        raise ValueError(f"{label} contains {dup_n:,} duplicated sample_id groups")

# Verify key/date identity in the feature tables, not just sample_id uniqueness.
era_mismatch = int(scalar(f"""
    SELECT COUNT(*) FROM {parquet_scan(CANDIDATES_PATH)} c
    JOIN {parquet_scan(ERA5_PATH)} e USING (sample_id)
    WHERE CAST(c.station_id AS VARCHAR) IS DISTINCT FROM CAST(e.station_id AS VARCHAR)
       OR CAST(c.sample_date AS DATE) IS DISTINCT FROM CAST(e.sample_date AS DATE)
"""))
smap_mismatch = int(scalar(f"""
    SELECT COUNT(*) FROM {parquet_scan(CANDIDATES_PATH)} c
    JOIN {parquet_scan(SMAP_PATH)} s USING (sample_id)
    WHERE CAST(c.station_id AS VARCHAR) IS DISTINCT FROM CAST(s.station_id AS VARCHAR)
       OR CAST(c.sample_date AS DATE) IS DISTINCT FROM CAST(s.sample_date AS DATE)
"""))
if era_mismatch or smap_mismatch:
    raise ValueError(f"Feature key mismatch: ERA5={era_mismatch:,}, SMAP={smap_mismatch:,}")

# Build and save the yearly raster index without loading all years at once.
raster_index_stage = STAGING_DIR / "raster_index.parquet"
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{sql_path(STAGING_DIR / 'raster_index_*.parquet')}')
    ) TO '{sql_path(raster_index_stage)}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
""")

# Build one compact row per sample_id; raster arrays remain external and lazily readable.
sample_index_stage = STAGING_DIR / "sample_index.parquet"
con.execute(f"""
COPY (
    WITH c AS (
        SELECT *, CAST(sample_date AS DATE) AS _date
        FROM {parquet_scan(CANDIDATES_PATH)}
    ),
    e AS (
        SELECT sample_id, era5_point_id, era5_lat, era5_lon,
               CAST(t2m_K AS DOUBLE) - 273.15 AS t2m_C,
               CAST(d2m_K AS DOUBLE) - 273.15 AS d2m_C,
               CAST(precip_mm AS DOUBLE) AS precip_mm,
               CAST(VPD_kPa AS DOUBLE) AS VPD_kPa,
               CAST(API_mm AS DOUBLE) AS API_mm
        FROM {parquet_scan(ERA5_PATH)}
    ),
    s AS (
        SELECT sample_id, CAST(SMAP AS DOUBLE) AS SMAP,
               SMAP_n_obs, SMAP_cell_conflict
        FROM {parquet_scan(SMAP_PATH)}
    ),
    si AS (SELECT * FROM {parquet_scan(soil_stage)}),
    te AS (SELECT * FROM {parquet_scan(terrain_stage)}),
    ri AS (SELECT * FROM {parquet_scan(raster_index_stage)})
    SELECT
        c.sample_id,
        CAST(c.station_id AS VARCHAR) AS station_id,
        CAST(c.network AS VARCHAR) AS network,
        CAST(c.network_name AS VARCHAR) AS network_name,
        CAST(c.station AS VARCHAR) AS station,
        CAST(c.network AS VARCHAR) || '::' || CAST(c.station_id AS VARCHAR) AS station_key,
        c._date AS sample_date,
        YEAR(c._date)::SMALLINT AS year,
        DAYOFYEAR(c._date)::SMALLINT AS day_of_year,
        SIN(2 * PI() * DAYOFYEAR(c._date) / 365.25) AS day_sin,
        COS(2 * PI() * DAYOFYEAR(c._date) / 365.25) AS day_cos,
        CAST(c.latitude AS DOUBLE) AS latitude,
        CAST(c.longitude AS DOUBLE) AS longitude,
        CAST(c.elevation_m AS DOUBLE) AS station_elevation_m,
        CAST(c.climate_zone AS VARCHAR) AS climate_zone,
        c.source_row_count,
        c.min_depth_from_m,
        c.max_depth_to_m,
        c.sensors,
        c.depth_classes,

        e.era5_point_id, e.era5_lat, e.era5_lon,
        e.t2m_C, e.d2m_C, e.precip_mm, e.VPD_kPa, e.API_mm,
        s.SMAP, s.SMAP_n_obs, s.SMAP_cell_conflict,

        si.clay_pct, si.sand_pct, si.silt_pct,
        te.elevation_m AS terrain_elevation_m,
        te.slope_deg, te.aspect_deg,

        ri.modis_row_index,
        ri.s1_row_index,
        (e.sample_id IS NOT NULL) AS has_era5_row,
        (s.sample_id IS NOT NULL AND s.SMAP IS NOT NULL) AS has_smap_value,
        (si.station_id IS NOT NULL) AS has_static_soil,
        (te.station_id IS NOT NULL) AS has_static_terrain
    FROM c
    LEFT JOIN e USING (sample_id)
    LEFT JOIN s USING (sample_id)
    LEFT JOIN si ON CAST(c.station_id AS VARCHAR) = CAST(si.station_id AS VARCHAR)
                AND CAST(c.network AS VARCHAR) = CAST(si.network AS VARCHAR)
    LEFT JOIN te ON CAST(c.station_id AS VARCHAR) = CAST(te.station_id AS VARCHAR)
                AND CAST(c.network AS VARCHAR) = CAST(te.network AS VARCHAR)
    LEFT JOIN ri USING (sample_id)
) TO '{sql_path(sample_index_stage)}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
""")

sample_index_rows = int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)}"))
sample_index_ids = int(scalar(f"SELECT COUNT(DISTINCT sample_id) FROM {parquet_scan(sample_index_stage)}"))
if sample_index_rows != candidate_count or sample_index_ids != candidate_count:
    raise ValueError(
        f"Sample-index cardinality changed during joins: rows={sample_index_rows:,}, distinct IDs={sample_index_ids:,}, expected={candidate_count:,}"
    )

print("Sample index rows:", f"{sample_index_rows:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample index rows: 3,514,643


## Target-observation table

The label file has more rows than the canonical sample index because a sample date can have multiple in-situ observations, including different sensors and depths. To avoid defining a scientifically questionable target in this preprocessing step, we keep every target row and link it to its `sample_id`.

The notebook preserves every row in `target_observations.parquet` and creates a separate `daily_targets.parquet` with the agreed Option A target: the mean of valid finite soil-moisture values linked to each `sample_id`, across all sensors/depths represented for that sample. This is a practical baseline, not a claim that different depth classes measure the same quantity. Counts, min/max/range, sensor/depth summaries, and invalid-value counts are retained for QC. `target_sample_summary.parquet` remains a separate diagnostic summary.

In [8]:
# ========================= LINK ALL TARGET OBSERVATIONS =========================
target_stage = STAGING_DIR / "target_observations.parquet"
label_select = [
    "station_id", "network", "network_name", "station", "latitude", "longitude",
    "elevation_m", "sensor", "depth_from_m", "depth_to_m", "depth_class",
    "sample_date", "soil_moisture_m3_m3", "observation_count", "source_file",
    "clay_pct", "sand_pct", "silt_pct", "climate_zone",
]
existing_label_select = [c for c in label_select if c in label_cols]
select_expr = [
    "c.sample_id",
    "CAST(l.station_id AS VARCHAR) AS station_id",
    "CAST(l.network AS VARCHAR) AS network",
    "CAST(l.sample_date AS DATE) AS sample_date",
    "CAST(l.soil_moisture_m3_m3 AS DOUBLE) AS soil_moisture_m3_m3",
]
for col in existing_label_select:
    if col in {"station_id", "network", "sample_date", "soil_moisture_m3_m3"}:
        continue
    select_expr.append(f"l.\"{col}\" AS \"{col}\"")

con.execute(f"""
COPY (
    SELECT {', '.join(select_expr)}
    FROM {parquet_scan(LABELS_PATH)} l
    INNER JOIN {parquet_scan(CANDIDATES_PATH)} c
        ON CAST(l.station_id AS VARCHAR) = CAST(c.station_id AS VARCHAR)
       AND CAST(l.network AS VARCHAR) = CAST(c.network AS VARCHAR)
       AND CAST(l.sample_date AS DATE) = CAST(c.sample_date AS DATE)
) TO '{sql_path(target_stage)}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
""")

label_rows = int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(LABELS_PATH)}"))
linked_target_rows = int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(target_stage)}"))
unlinked_label_rows = int(scalar(f"""
    SELECT COUNT(*) FROM {parquet_scan(LABELS_PATH)} l
    LEFT JOIN {parquet_scan(CANDIDATES_PATH)} c
      ON CAST(l.station_id AS VARCHAR) = CAST(c.station_id AS VARCHAR)
     AND CAST(l.network AS VARCHAR) = CAST(c.network AS VARCHAR)
     AND CAST(l.sample_date AS DATE) = CAST(c.sample_date AS DATE)
    WHERE c.sample_id IS NULL
"""))
if unlinked_label_rows:
    print(f"WARNING: {unlinked_label_rows:,} label rows did not link to a canonical sample.")
if linked_target_rows != label_rows - unlinked_label_rows:
    raise ValueError("Target linking count is internally inconsistent")

summary_stage = STAGING_DIR / "target_sample_summary.parquet"
con.execute(f"""
COPY (
    SELECT sample_id,
           COUNT(*) AS target_observation_count,
           AVG(soil_moisture_m3_m3) AS target_mean_qc_only,
           MIN(soil_moisture_m3_m3) AS target_min_qc_only,
           MAX(soil_moisture_m3_m3) AS target_max_qc_only,
           MIN(depth_from_m) AS min_depth_from_m_qc,
           MAX(depth_to_m) AS max_depth_to_m_qc,
           COUNT(DISTINCT depth_class) AS n_depth_classes,
           COUNT(DISTINCT sensor) AS n_sensors
    FROM {parquet_scan(target_stage)}
    GROUP BY sample_id
) TO '{sql_path(summary_stage)}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
""")

print({
    "source_label_rows": label_rows,
    "linked_target_observation_rows": linked_target_rows,
    "unlinked_label_rows": unlinked_label_rows,
    "samples_with_at_least_one_target": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(summary_stage)}")),
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'source_label_rows': 5045849, 'linked_target_observation_rows': 5045849, 'unlinked_label_rows': 0, 'samples_with_at_least_one_target': 3514643}


In [9]:
# ========================= QC REPORT =========================
feature_numeric_cols = [
    "t2m_C", "d2m_C", "precip_mm", "VPD_kPa", "API_mm", "SMAP",
    "clay_pct", "sand_pct", "silt_pct", "terrain_elevation_m", "slope_deg", "aspect_deg",
]
missingness = {}
for col in feature_numeric_cols:
    missingness[col] = int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)} WHERE \"{col}\" IS NULL OR NOT isfinite(\"{col}\")"))

# Candidate rows are only on label/sample dates, which may not be consecutive days.
# This gap summary helps the training notebook decide whether histories are truly daily.
gap_stats = con.execute(f"""
    WITH ordered AS (
        SELECT CAST(sample_date AS DATE) AS sample_date,
               LAG(CAST(sample_date AS DATE)) OVER (
                   PARTITION BY network, station_id
                   ORDER BY CAST(sample_date AS DATE), sample_id
               ) AS previous_date
        FROM {parquet_scan(CANDIDATES_PATH)}
    ), gaps AS (
        SELECT DATE_DIFF('day', previous_date, sample_date) AS gap_days
        FROM ordered WHERE previous_date IS NOT NULL
    )
    SELECT COUNT(*) AS n_gaps,
           MEDIAN(gap_days) AS median_gap_days,
           QUANTILE_CONT(gap_days, 0.90) AS p90_gap_days,
           QUANTILE_CONT(gap_days, 0.95) AS p95_gap_days,
           AVG(CASE WHEN gap_days = 1 THEN 1.0 ELSE 0.0 END) AS fraction_1_day_gaps,
           AVG(CASE WHEN gap_days <= 7 THEN 1.0 ELSE 0.0 END) AS fraction_gaps_le_7_days
    FROM gaps
""").df().iloc[0].to_dict()
gap_stats = {k: (None if pd.isna(v) else float(v) if k != "n_gaps" else int(v)) for k, v in gap_stats.items()}

qc = {
    "root": str(ROOT),
    "output_dir": str(OUTPUT_DIR),
    "years": YEARS,
    "canonical_sample_rows": candidate_count,
    "sample_index_rows": sample_index_rows,
    "sample_index_distinct_ids": sample_index_ids,
    "source_label_rows": label_rows,
    "linked_target_observation_rows": linked_target_rows,
    "unlinked_label_rows": unlinked_label_rows,
    "samples_with_target": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(summary_stage)}")),
    "feature_table_missing_or_nonfinite_counts": missingness,
    "candidate_date_gap_summary_days": gap_stats,
    "sample_rows_missing_era5_join": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)} WHERE NOT has_era5_row")),
    "smap_values_missing": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)} WHERE SMAP IS NULL OR NOT isfinite(SMAP)")),
    "smap_conflict_flagged_rows": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)} WHERE SMAP_cell_conflict IS TRUE")),
    "rows_missing_static_soil": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)} WHERE NOT has_static_soil")),
    "rows_missing_static_terrain": int(scalar(f"SELECT COUNT(*) FROM {parquet_scan(sample_index_stage)} WHERE NOT has_static_terrain")),
    "raster_years_all_passed": bool(raster_audit["passed"].all()),
    "target_policy": "preserve_all_observations and create explicit Option A daily mean target per sample_id across valid observations; sensor/depth metadata retained",
    "normalization_or_imputation_applied": False,
}

print(json.dumps(qc, indent=2, default=str))

{
  "root": "/home/mayur/Code/UG_SM/Data/Global",
  "output_dir": "/home/mayur/Code/UG_SM/Data/Global/model_ready_global_v1",
  "years": [
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
  ],
  "canonical_sample_rows": 3514643,
  "sample_index_rows": 3514643,
  "sample_index_distinct_ids": 3514643,
  "source_label_rows": 5045849,
  "linked_target_observation_rows": 5045849,
  "unlinked_label_rows": 0,
  "samples_with_target": 3514643,
  "feature_table_missing_or_nonfinite_counts": {
    "t2m_C": 25258,
    "d2m_C": 25258,
    "precip_mm": 1879,
    "VPD_kPa": 25258,
    "API_mm": 1879,
    "SMAP": 3795,
    "clay_pct": 78371,
    "sand_pct": 78371,
    "silt_pct": 78371,
    "terrain_elevation_m": 0,
    "slope_deg": 0,
    "aspect_deg": 0
  },
  "candidate_date_gap_summary_days": {
    "n_gaps": 3512433,
    "median_gap_days": 1.0,
    "p90_gap_days": 1.0,
    "p95_gap_days": 1.0,
    "fraction_1_day_gaps": 0.97973

In [10]:
# ========================= WRITE FINAL ARTIFACTS =========================
feature_schema = {
    "schema_version": "1.0",
    "dataset_name": "Global MultiModalSoilNet model-ready index",
    "sample_unit": "one canonical station-date candidate per sample_id",
    "target_unit": "daily_targets.parquet has one Option A mean target per sample_id; all source observations remain in target_observations.parquet",
    "identity_columns": ["sample_id", "station_id", "network", "station_key", "sample_date", "year"],
    "metadata_not_default_model_features": ["latitude", "longitude", "climate_zone", "station_id", "network", "station_key", "source_row_count", "sensors", "depth_classes"],
    "spatial_modalities": {
        "modis": {
            "store_template": "modis/evi_lst_patch_dataset/{year}/patches.zarr",
            "row_index_column": "modis_row_index",
            "row_index_basis": "producer sorts year_samples by sample_date, sample_id before writing; sidecar physical order is not used",
            "arrays": {"evi": {"shape": [27, 27]}, "lst": {"shape": [13, 13]}},
        },
        "sentinel1": {
            "store_template": "sentinel1/nearest_valid_overview_v1/s1_nearest_valid_{year}.zarr",
            "row_index_column": "s1_row_index",
            "arrays": {
                "vv": {"shape": [13, 13]}, "vh": {"shape": [13, 13]},
                "hh": {"shape": [13, 13]}, "hv": {"shape": [13, 13]},
                "vv_vh_ratio": {"shape": [13, 13]},
                "source_day_offset": {"shape": [2]}, "obs_count": {"shape": [4]},
                "sample_done": {"shape": []},
            },
        },
        "alignment_note": "MODIS row indices are reconstructed from the producer documented sample_date/sample_id sort, not sidecar physical order. This assumes the current Zarr rows were written/resumed under that same sort contract; a historical writer-order change requires rebuilding MODIS arrays/metadata. EVI is 27x27; LST and Sentinel-1 are 13x13. Use separate modality stems or a documented geospatial alignment step; do not directly stack as channels.",
    },
    "temporal_candidate_features": ["precip_mm", "t2m_C", "d2m_C", "VPD_kPa", "API_mm"],
    "temporal_calendar_features": ["day_of_year", "day_sin", "day_cos"],
    "static_candidate_features": ["clay_pct", "sand_pct", "silt_pct", "terrain_elevation_m", "slope_deg", "aspect_deg"],
    "anchor_feature": "SMAP",
    "anchor_metadata": ["SMAP_n_obs", "SMAP_cell_conflict", "has_smap_value"],
    "target_column": "target_mean_m3_m3",
    "target_columns_to_review": ["sensor", "depth_from_m", "depth_to_m", "depth_class", "observation_count"],
    "feature_units": {
        "t2m_C": "degrees Celsius", "d2m_C": "degrees Celsius", "precip_mm": "mm per day as supplied by ERA5 export",
        "VPD_kPa": "kPa", "API_mm": "mm", "SMAP": "m3/m3 as supplied by SMAP join",
        "clay_pct": "percent", "sand_pct": "percent", "silt_pct": "percent",
        "terrain_elevation_m": "meters", "slope_deg": "degrees", "aspect_deg": "degrees",
    },
    "preprocessing_policy": {
        "imputation": "none; preserve nulls and quality flags",
        "normalization": "none; compute using training partition only in training notebook",
        "label_aggregation": "Option A: arithmetic mean of valid finite soil_moisture_m3_m3 observations per sample_id across available sensors/depths; raw observations retained and depth-mixing metadata recorded",
        "split": "not created here; training notebook handles environmental-only and location-aware split modes",
    },
}

readme = f"""# Global MultiModalSoilNet model-ready data\n\nBuilt from: `{ROOT}`\n\n## Files\n- `sample_index.parquet`: one row per canonical sample_id, joined tabular features and raster row pointers.\n- `target_observations.parquet`: all linked in-situ observations, preserved unchanged.\n- `daily_targets.parquet`: one Option A mean target per sample_id plus valid/invalid counts, range, sensor and depth QC metadata. This baseline can mix depths; use raw observations for depth-restricted sensitivity analyses.\n- `target_sample_summary.parquet`: additional QC summary of label multiplicity/ranges.\n- `raster_alignment_audit.parquet`: annual raster alignment/shape/completion/read checks.\n- `feature_schema.json`: feature and array contracts.\n- `prep_qc.json`: join coverage and missingness counts.\n\n## Loading raster data lazily\nFor a row with `year`, `modis_row_index`, and `s1_row_index`, open the MODIS store at `modis/evi_lst_patch_dataset/{{year}}/patches.zarr` and the Sentinel-1 store at `sentinel1/nearest_valid_overview_v1/s1_nearest_valid_{{year}}.zarr`. Read only the required row from each Zarr array; do not materialize all patches at once.\n\n## Model feature routing\n- Spatial branch: MODIS EVI/LST patches and Sentinel-1 patches. EVI is 27x27 while LST/Sentinel-1 are 13x13, so use separate stems or geospatially align them deliberately.\n- Temporal branch candidate columns: `precip_mm`, `t2m_C`, `d2m_C`, `VPD_kPa`, `API_mm`, plus calendar features. Note that ERA5 here is joined to candidate sample dates; inspect real day gaps before constructing history windows.\n- Static branch candidate columns: clay/sand/silt percentages, terrain elevation, slope, aspect.\n- Current-date residual anchor: `SMAP`, with `SMAP_n_obs` and `SMAP_cell_conflict` retained as metadata.\n- Default target: `target_mean_m3_m3` from `daily_targets.parquet`. Review its depth/sensor metadata because Option A may combine observations across depth classes; `target_observations.parquet` retains the underlying rows.\n\n## Important\nNo feature imputation, normalization, train/test splitting, raster copying, or calibration occurs here. Target aggregation is explicit Option A in `daily_targets.parquet`; raw target observations are retained and can support depth-restricted alternatives.\n"""

# Audit as a human-readable Parquet table and JSON records.
audit_stage = STAGING_DIR / "raster_alignment_audit.parquet"
raster_audit.to_parquet(audit_stage, index=False, compression="zstd")

artifacts_to_publish = {
    "sample_index.parquet": sample_index_stage,
    "target_observations.parquet": target_stage,
    "target_sample_summary.parquet": summary_stage,
    "raster_alignment_audit.parquet": audit_stage,
}
for final_name, staged_path in artifacts_to_publish.items():
    final_path = OUTPUT_DIR / final_name
    if final_path.exists() and not REBUILD_OUTPUT:
        raise FileExistsError(f"Refusing to overwrite {final_path}; set REBUILD_OUTPUT=True to replace it.")
    shutil.copy2(staged_path, final_path)

for name, payload in [("feature_schema.json", feature_schema), ("prep_qc.json", qc)]:
    target_path = OUTPUT_DIR / name
    if target_path.exists() and not REBUILD_OUTPUT:
        raise FileExistsError(f"Refusing to overwrite {target_path}; set REBUILD_OUTPUT=True to replace it.")
    target_path.write_text(json.dumps(payload, indent=2, default=str) + "\n", encoding="utf-8")

readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists() and not REBUILD_OUTPUT:
    raise FileExistsError(f"Refusing to overwrite {readme_path}; set REBUILD_OUTPUT=True to replace it.")
readme_path.write_text(readme, encoding="utf-8")

print("\nPublished model-ready files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:36s} {path.stat().st_size / (1024**2):8.2f} MiB")


Published model-ready files:
README.md                                0.00 MiB
feature_schema.json                      0.00 MiB
prep_qc.json                             0.00 MiB
raster_alignment_audit.parquet           0.01 MiB
sample_index.parquet                    97.49 MiB
target_observations.parquet             26.35 MiB
target_sample_summary.parquet           41.32 MiB


In [11]:
# ========================= FINAL SMOKE TEST =========================
# Verify that the first linked training example can be resolved to tabular data,
# one target row, and readable rasters without loading the dataset into RAM.
final_index = OUTPUT_DIR / "sample_index.parquet"
final_targets = OUTPUT_DIR / "target_observations.parquet"

smoke = con.execute(f"""
    SELECT i.sample_id, i.year, i.station_id, i.network, i.sample_date,
           i.modis_row_index, i.s1_row_index, i.SMAP,
           t.soil_moisture_m3_m3, t.depth_from_m, t.depth_to_m, t.sensor
    FROM {parquet_scan(final_index)} i
    JOIN {parquet_scan(final_targets)} t USING (sample_id)
    ORDER BY i.sample_id
    LIMIT 1
""").df()
if smoke.empty:
    raise RuntimeError("Smoke test found no linked sample/target pair")
display(smoke)
r = smoke.iloc[0]
year = int(r["year"])
modis_group = open_group(MODIS_ROOT / str(year) / "patches.zarr")
s1_group = open_group(S1_ROOT / f"s1_nearest_valid_{year}.zarr")
mi = int(r["modis_row_index"])
si = int(r["s1_row_index"])

sample_patches = {
    "EVI": np.asarray(modis_group[MODIS_ARRAY_NAMES["evi"]][mi]),
    "LST": np.asarray(modis_group[MODIS_ARRAY_NAMES["lst"]][mi]),
    "S1_VV": np.asarray(s1_group[S1_ARRAY_NAMES["vv"]][si]),
    "S1_VH": np.asarray(s1_group[S1_ARRAY_NAMES["vh"]][si]),
    "S1_VV_VH_ratio": np.asarray(s1_group[S1_ARRAY_NAMES["vv_vh_ratio"]][si]),
}
print("Raster sample shapes:", {k: tuple(v.shape) for k, v in sample_patches.items()})
assert sample_patches["EVI"].shape == (27, 27)
assert sample_patches["LST"].shape == (13, 13)
assert sample_patches["S1_VV"].shape == (13, 13)
print("Smoke test passed: sample_id -> table row -> target row -> yearly Zarr patches.")

,sample_id,year,station_id,network,sample_date,modis_row_index,s1_row_index,SMAP,soil_moisture_m3_m3,depth_from_m,depth_to_m,sensor
0,0,2015,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-01,0,93115,0.022373,0.017354,0.05,0.05,Campbell-CS616


Raster sample shapes: {'EVI': (27, 27), 'LST': (13, 13), 'S1_VV': (13, 13), 'S1_VH': (13, 13), 'S1_VV_VH_ratio': (13, 13)}
Smoke test passed: sample_id -> table row -> target row -> yearly Zarr patches.


## Before starting the training notebook

1. Review `prep_qc.json` and `raster_alignment_audit.parquet`.
2. The default baseline target is `daily_targets.parquet` (`target_mean_m3_m3`), one mean per `sample_id`. It may mix sensors/depth classes; review `n_depth_classes`, depth bounds, sensors and target range. Keep `target_observations.parquet` for depth-restricted sensitivity analyses.
3. Check the distribution of date gaps per station. The ERA5 table is keyed to candidate sample dates, so it should not be assumed to be a complete daily weather archive.
4. In the training notebook, define the two split modes before fitting scalers or building cached tensors: (a) environment-only inputs, and (b) location/history-aware inputs. Fit normalization, imputation, station statistics and calibration using only data permitted by the split.
5. Build batches by reading only the relevant Zarr rows. Do not load all annual patch tensors into RAM.
6. For modeling, join `sample_index.parquet` to `daily_targets.parquet` on `sample_id`; do not use `target_sample_summary.parquet` as the training target.

## Option A — create one daily target per `sample_id`

This step is deliberately separate from the raw target-observation table. It can be run by itself after Notebook 1 has already produced `model_ready_global_v1/target_observations.parquet`; you do **not** need to rebuild the raster index or rerun the earlier preparation cells.

For each `sample_id`, `target_mean_m3_m3` is the arithmetic mean of all non-null, finite `soil_moisture_m3_m3` observations linked to that sample. This baseline may average across sensor and depth classes. The raw observations remain unchanged, and the output stores target counts, minimum/maximum/range/standard deviation, sensor/depth summaries, plus invalid-value counts for review.


In [13]:
# ========================= OPTION A DAILY TARGET AGGREGATION =========================
# Memory-conscious replacement cell.
# Reads target observations from Parquet, aggregates with DuckDB, and writes
# one daily target per sample_id. The raw observations are never modified.

from pathlib import Path
import json
import os
import shutil
import duckdb

from IPython.display import display

# ---------- Configuration ----------
ROOT = Path.home() / "Code" / "UG_SM" / "Data" / "Global"
OUTPUT_DIR = ROOT / "model_ready_global_v1"

SOURCE_TARGETS = OUTPUT_DIR / "target_observations.parquet"
DAILY_TARGETS = OUTPUT_DIR / "daily_targets.parquet"

WORK_DIR = OUTPUT_DIR / "_build_tmp"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TEMP_OUTPUT = WORK_DIR / "daily_targets_option_a_tmp.parquet"
SPILL_DIR = WORK_DIR / "duckdb_spill_option_a"

# Lower this if RAM is very limited; increase only if memory is available.
DUCKDB_MEMORY_LIMIT = "1GB"
DUCKDB_THREADS = 2

if not SOURCE_TARGETS.exists():
    raise FileNotFoundError(
        f"Cannot find source observations:\n{SOURCE_TARGETS}\n"
        "Run the preparation notebook cells first."
    )

# Allow a rerun after an interrupted/failed previous attempt.
if TEMP_OUTPUT.exists():
    TEMP_OUTPUT.unlink()

shutil.rmtree(SPILL_DIR, ignore_errors=True)
SPILL_DIR.mkdir(parents=True, exist_ok=True)


def sql_path(path: Path) -> str:
    """Format a path safely as a DuckDB SQL string literal."""
    return str(path.resolve()).replace("'", "''")


source_sql = sql_path(SOURCE_TARGETS)
temp_output_sql = sql_path(TEMP_OUTPUT)
spill_sql = sql_path(SPILL_DIR)

print("Input:", SOURCE_TARGETS)
print("Output:", DAILY_TARGETS)
print("DuckDB memory limit:", DUCKDB_MEMORY_LIMIT)
print("DuckDB threads:", DUCKDB_THREADS)
print("Spill directory:", SPILL_DIR)

# ---------- Stream the Parquet aggregation through DuckDB ----------
con = duckdb.connect(database=":memory:")

try:
    con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
    con.execute(f"SET threads = {DUCKDB_THREADS}")
    con.execute("SET preserve_insertion_order = false")
    con.execute(f"SET temp_directory = '{spill_sql}'")

    con.execute(f"""
        COPY (
            WITH src AS (
                SELECT
                    CAST(sample_id AS VARCHAR) AS sample_id,
                    CAST(station_id AS VARCHAR) AS station_id,
                    CAST(network AS VARCHAR) AS network,
                    CAST(sample_date AS DATE) AS sample_date,
                    TRY_CAST(soil_moisture_m3_m3 AS DOUBLE) AS y,
                    TRY_CAST(depth_from_m AS DOUBLE) AS depth_from_m,
                    TRY_CAST(depth_to_m AS DOUBLE) AS depth_to_m,
                    NULLIF(TRIM(CAST(depth_class AS VARCHAR)), '') AS depth_class,
                    NULLIF(TRIM(CAST(sensor AS VARCHAR)), '') AS sensor
                FROM read_parquet('{source_sql}')
            )
            SELECT
                sample_id,

                ANY_VALUE(station_id) AS station_id,
                ANY_VALUE(network) AS network,
                MIN(sample_date) AS sample_date,

                COUNT(*) AS linked_observation_count,

                COUNT(*) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                ) AS target_valid_observation_count,

                COUNT(*) FILTER (
                    WHERE y IS NULL OR NOT isfinite(y)
                ) AS target_invalid_observation_count,

                AVG(y) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                ) AS target_mean_m3_m3,

                MIN(y) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                ) AS target_min_m3_m3,

                MAX(y) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                ) AS target_max_m3_m3,

                MAX(y) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                )
                - MIN(y) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                ) AS target_range_m3_m3,

                STDDEV_SAMP(y) FILTER (
                    WHERE y IS NOT NULL AND isfinite(y)
                ) AS target_std_m3_m3,

                MIN(depth_from_m) AS min_depth_from_m,
                MAX(depth_to_m) AS max_depth_to_m,

                COUNT(DISTINCT depth_class) AS n_depth_classes,
                COUNT(DISTINCT sensor) AS n_sensors,

                STRING_AGG(DISTINCT sensor, ', ') AS sensors,
                STRING_AGG(DISTINCT depth_class, ', ') AS depth_classes,

                COUNT(DISTINCT station_id) AS n_station_ids_in_group,
                COUNT(DISTINCT network) AS n_networks_in_group

            FROM src
            GROUP BY sample_id
        )
        TO '{temp_output_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
    """)

finally:
    con.close()

# Publish only after the aggregation query succeeds.
# Replaces an earlier daily_targets.parquet but never changes the raw observations.
os.replace(TEMP_OUTPUT, DAILY_TARGETS)

# ---------- Verify output without loading the full file into memory ----------
check = duckdb.connect(database=":memory:")

try:
    check.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
    check.execute(f"SET temp_directory = '{spill_sql}'")
    check.execute(f"SET threads = {DUCKDB_THREADS}")

    summary = check.execute(f"""
        SELECT
            COUNT(*) AS daily_target_rows,
            COUNT(DISTINCT sample_id) AS distinct_sample_ids,
            COUNT(*) FILTER (
                WHERE target_mean_m3_m3 IS NULL
            ) AS rows_without_valid_target,
            COUNT(*) FILTER (
                WHERE n_depth_classes > 1
            ) AS rows_mixing_depth_classes,
            COUNT(*) FILTER (
                WHERE n_sensors > 1
            ) AS rows_with_multiple_sensors,
            MAX(linked_observation_count) AS max_linked_observations_per_sample
        FROM read_parquet('{sql_path(DAILY_TARGETS)}')
    """).fetchone()

    summary_names = [
        "daily_target_rows",
        "distinct_sample_ids",
        "rows_without_valid_target",
        "rows_mixing_depth_classes",
        "rows_with_multiple_sensors",
        "max_linked_observations_per_sample",
    ]
    summary_dict = dict(zip(summary_names, summary))

    if summary_dict["daily_target_rows"] != summary_dict["distinct_sample_ids"]:
        raise RuntimeError(
            "Validation failed: output does not contain exactly one row per sample_id."
        )

    print("\nAggregation QC:")
    for key, value in summary_dict.items():
        print(f"{key}: {value}")

    # Show only a small preview.
    preview = check.execute(f"""
        SELECT
            sample_id,
            linked_observation_count,
            target_valid_observation_count,
            target_mean_m3_m3,
            target_min_m3_m3,
            target_max_m3_m3,
            n_sensors,
            sensors,
            n_depth_classes,
            depth_classes,
            min_depth_from_m,
            max_depth_to_m
        FROM read_parquet('{sql_path(DAILY_TARGETS)}')
        ORDER BY linked_observation_count DESC, sample_id
        LIMIT 10
    """).fetchdf()

    display(preview)

finally:
    check.close()

# ---------- Update small metadata files, if they exist ----------
qc_path = OUTPUT_DIR / "prep_qc.json"

if qc_path.exists():
    qc_payload = json.loads(qc_path.read_text(encoding="utf-8"))
    qc_payload["target_policy"] = (
        "Option A: arithmetic mean of valid finite soil_moisture_m3_m3 "
        "observations per sample_id across available sensors/depths; "
        "raw observations preserved"
    )
    qc_payload["daily_target_output"] = DAILY_TARGETS.name
    qc_payload["daily_target_rows"] = int(summary_dict["daily_target_rows"])
    qc_payload["daily_target_rows_without_valid_target"] = int(
        summary_dict["rows_without_valid_target"]
    )
    qc_path.write_text(
        json.dumps(qc_payload, indent=2, default=str) + "\n",
        encoding="utf-8",
    )

schema_path = OUTPUT_DIR / "feature_schema.json"

if schema_path.exists():
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    schema["target_unit"] = (
        "One aggregated target per sample_id in daily_targets.parquet; "
        "all original observations remain in target_observations.parquet"
    )
    schema["target_column"] = "target_mean_m3_m3"
    schema["target_columns_to_review"] = [
        "linked_observation_count",
        "target_valid_observation_count",
        "target_invalid_observation_count",
        "target_min_m3_m3",
        "target_max_m3_m3",
        "target_range_m3_m3",
        "target_std_m3_m3",
        "sensors",
        "depth_classes",
        "min_depth_from_m",
        "max_depth_to_m",
        "n_sensors",
        "n_depth_classes",
    ]
    schema.setdefault("preprocessing_policy", {})["label_aggregation"] = (
        "Option A: arithmetic mean of valid finite observations per sample_id "
        "across linked sensors/depths; source observations are preserved"
    )
    schema_path.write_text(
        json.dumps(schema, indent=2, default=str) + "\n",
        encoding="utf-8",
    )

readme_path = OUTPUT_DIR / "README.md"

if readme_path.exists():
    readme = readme_path.read_text(encoding="utf-8")
    if "`daily_targets.parquet`" not in readme:
        readme += (
            "\n\n## Daily target aggregation\n"
            "- `daily_targets.parquet`: one Option A mean target per sample_id, "
            "with observation counts, target range, and sensor/depth QC metadata.\n"
            "- This baseline can average across depths. Use "
            "`target_observations.parquet` for depth-restricted analyses.\n"
        )
        readme_path.write_text(readme, encoding="utf-8")

print("\nSUCCESS")
print("Created:", DAILY_TARGETS)
print("Output size (MiB):", round(DAILY_TARGETS.stat().st_size / (1024**2), 2))
print("Original target_observations.parquet was not modified.")

# Clean up spill files after successful completion.
shutil.rmtree(SPILL_DIR, ignore_errors=True)

Input: /home/mayur/Code/UG_SM/Data/Global/model_ready_global_v1/target_observations.parquet
Output: /home/mayur/Code/UG_SM/Data/Global/model_ready_global_v1/daily_targets.parquet
DuckDB memory limit: 1GB
DuckDB threads: 2
Spill directory: /home/mayur/Code/UG_SM/Data/Global/model_ready_global_v1/_build_tmp/duckdb_spill_option_a


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Aggregation QC:
daily_target_rows: 3514643
distinct_sample_ids: 3514643
rows_without_valid_target: 0
rows_mixing_depth_classes: 820577
rows_with_multiple_sensors: 139744
max_linked_observations_per_sample: 12


,sample_id,linked_observation_count,target_valid_observation_count,target_mean_m3_m3,target_min_m3_m3,target_max_m3_m3,n_sensors,sensors,n_depth_classes,depth_classes,min_depth_from_m,max_depth_to_m
0,864569,12,12,0.192297,0.101000,0.241750,1,Stevens-Hydraprobe-Digital,1,near_surface,0.000,0.05
1,888805,12,12,0.096583,0.074200,0.120500,1,Stevens-Hydraprobe-Digital,1,near_surface,0.000,0.05
2,42810,11,11,0.218157,0.050100,0.371314,3,"Campbell-229, SMP1, Stevens-Hydraprobe-Digital",2,"fallback_deeper, near_surface",0.025,0.10
3,32566,9,9,0.289331,0.253400,0.335790,2,"Stevens-Hydraprobe-Digital, SMP1",2,"near_surface, fallback_deeper",0.025,0.10
4,32567,9,9,0.289155,0.255250,0.334200,2,"SMP1, Stevens-Hydraprobe-Digital",2,"near_surface, fallback_deeper",0.025,0.10
5,32592,9,9,0.288671,0.250180,0.336280,2,"Stevens-Hydraprobe-Digital, SMP1",2,"near_surface, fallback_deeper",0.025,0.10
6,32593,9,9,0.289033,0.249429,0.335725,2,"SMP1, Stevens-Hydraprobe-Digital",2,"fallback_deeper, near_surface",0.025,0.10
7,32594,9,9,0.288946,0.248300,0.336167,2,"Stevens-Hydraprobe-Digital, SMP1",2,"fallback_deeper, near_surface",0.025,0.10
8,32595,9,9,0.284437,0.250980,0.331573,2,"SMP1, Stevens-Hydraprobe-Digital",2,"near_surface, fallback_deeper",0.025,0.10
9,32596,9,9,0.283290,0.248900,0.329940,2,"Stevens-Hydraprobe-Digital, SMP1",2,"fallback_deeper, near_surface",0.025,0.10



SUCCESS
Created: /home/mayur/Code/UG_SM/Data/Global/model_ready_global_v1/daily_targets.parquet
Output size (MiB): 69.28
Original target_observations.parquet was not modified.
